https://skphd.medium.com/top-45-apache-spark-interview-questions-and-answers-da9a8f488d20
https://dev.to/hannah_usmedynska/100-spark-scenario-based-interview-questions-and-answers-344m


In [ ]:

# For file-based sources, Spark considers the file sizes and spark.sql.files.maxPartitionBytes, whose default is 128 MB in current Spark documentation.

# df = spark.read.parquet("/data/orders")

# print(df.rdd.getNumPartitions())

# Spark's own tuning documentation notes that you generally want enough tasks to keep the cluster utilized and gives 2–3 tasks per CPU core as a general guideline.


# Each partition in Apache Spark is processed by exactly one task.

# Parallelism Bound: Your maximum parallel execution is limited by whichever is smaller: your total available CPU cores or your number of partitions. 
If you have 64 CPU cores but only 4 partitions, Spark will only generate 4 tasks, leaving 60 cores idle.
CPU Cores: They dictate how many tasks can run simultaneously at any given split second. If you have 8 cores, you can run exactly 
8 tasks at the exact same time.
Tasks:  represent the total amount of work. The number of tasks is strictly determined by the number of partitions in your data. If your data 
has 200 partitions, Spark will create 200 tasks.

Driver: 
When Spark reads an file as a DataFrame, the file is divided into partitions and executors process those partitions in parallel. 
Driver only read metadata and obtains information needed to plan the read.



# Modern Spark has Adaptive Query Execution (AQE) enabled by default in current Spark releases. It uses runtime statistics to optimize the query.


# For example:

# spark.sql.shuffle.partitions = 200

# doesn't necessarily mean Spark will finally execute exactly 200 useful tasks.

# Imagine after the shuffle Spark discovers:

# Partition 1 = 10 MB
# Partition 2 = 8 MB
# Partition 3 = 12 MB
# ...

# Many partitions are tiny.

# AQE can coalesce small adjacent shuffle partitions.

# So conceptually:

# Initial:

# 200 shuffle partitions

#         ↓ AQE

# 80 effective partitions

# This prevents lots of unnecessarily small tasks

# So AQE can effectively:
# COALESCE small partitions
#            +
# SPLIT skewed partitions


# df = df.coalesce(20)

# This is primarily used to reduce the number of partitions without a full shuffle.

# df.repartition(20)
# redistributes the data through a shuffle.

# Shuffle operation:

# groupBy()
# join()
# distinct()
# orderBy()

# Spark generally creates shuffle partitions according to:
spark.sql.shuffle.partitions=200 


# "How does Spark decide the number of partitions?"

It depends on the stage. For file reads, Spark creates input partitions based largely on file sizes and file-splitting configuration such as spark.sql.files.maxPartitionBytes. 
For shuffle operations such as joins and aggregations, Spark initially uses spark.sql.shuffle.partitions.
In modern Spark, Adaptive Query Execution can then coalesce small shuffle partitions or split skewed partitions based on runtime statistics. 
The actual number of concurrently running tasks is constrained by the available executor cores.


# Job ➔ Stage ➔ Task
Here is exactly how a Stage breaks down into Tasks:
1. Job (The Big Picture)When you call an action in your code (like .count(), .collect(), or .save()), Spark triggers a Job. A single Spark application can run many jobs.
2. Stage (Dividing the Job)Spark looks at the execution plan (DAG) for that job and breaks it into Stages.Stages are separated by shuffle boundaries (wide transformations like .groupBy() or .join()).
If Spark can perform multiple transformations in memory without moving data across the network (like .map() and .filter()), it glues them together into a single stage.
3. Task (The Actual Work)Once a Stage is defined, Spark looks at how many data partitions exist for that stage. It then breaks the stage down into Tasks.
As established, Spark creates exactly one task per partition for that stage.The Stage cannot be completed until all of its individual Tasks have finished processing their respective partitions.

Action:

| Step                               | Happens when? |
| ---------------------------------- | ------------- |
| Transformations build logical plan | Before action |
| count() called                     | 🔴 Action     |
| Catalyst optimization optimized plan| After action  |
| Physical plan generated            | After action  |
| DAG/stages determined              | After action  |
| Tasks created                      | After action  |
| Executors execute tasks            | After action  |



# 1. Big picture
#                     Spark Application
#                            |
#                            ↓
#                       ┌─────────┐
#                       │ Driver  │
#                       └────┬────┘
#                            |
#                 asks cluster manager
#                            |
#           ┌────────────────┼────────────────┐
#           ↓                ↓                ↓
#     ┌──────────┐     ┌──────────┐     ┌──────────┐
#     │ Executor │     │ Executor │     │ Executor │
#     │    1     │     │    2     │     │    3     │
#     ├──────────┤     ├──────────┤     ├──────────┤
#     │ Task     │     │ Task     │     │ Task     │
#     │ Task     │     │ Task     │     │ Task     │
#     │ Task     │     │ Task     │     │ Task     │
#     └──────────┘     └──────────┘     └──────────┘

# The simple mental model:

# Driver coordinates. Executors do the actual data processing.

# 2. What is the Driver?
# The Driver is the process that runs your Spark application and coordinates the work. Driver operates strictly at the application layer.

It does things like:
I need an executor process with X amount of CPU , memory and storage."
create the Spark session/context
build the logical plan , optimized plan
create the physical execution plan
divide the work into stages
create tasks
schedule tasks on executors
monitor the execution of individual tasks and states within executor processes
coordinate failures/retries : The Driver handles logic-based and process-based retries:
Task Retries: If a task fails (e.g., due to a data fetch error or temporary network glitch), the Driver automatically schedules a retry of that specific task on a different executor, 
up to spark.task.maxFailures times.
Executor Retries: If an entire Executor process crashes (e.g., due to a Java OutOfMemoryError), the Driver talks to the Cluster Manager to request a new process executer container.

# But the Driver generally does not process the entire dataset itself.

# 3. What are Executors?

Executors are processes running on worker machines.

                    SPARK CLUSTER
                         │
        ┌────────────────┼────────────────┐
        │                │                │
        ▼                ▼                ▼
   WORKER 1          WORKER 2          WORKER 3
   CPU: 16 cores     CPU: 16 cores     CPU: 32 cores
   RAM: 64 GB        RAM: 64 GB        RAM: 128 GB
   Disk               Disk               Disk
      │                │                  │
      ├── Executor 1   ├── Executor 3     └── Executor 5
      │   4 cores      │   4 cores            8 cores
      │   16 GB        │   16 GB              32 GB
      │                │
      └── Executor 2   └── Executor 4
          4 cores          4 cores
          16 GB            16 GB

Executors perform the actual computation.

# They:

# read data
# execute tasks
# perform transformations
# keep cached data in memory/disk
# send results back as required
# maintain state for applicable Structured Streaming operations

# 4. TASK 
# This is where Spark's parallelism becomes important.
# A task is the unit of work that processes one partition for a particular stage.

# 5. What is a Partition?
# A partition is a chunk of distributed data.

# 6. Who actually starts the executors?
# This is where the Cluster Manager comes in. The cluster manager is responsible for allocating resources for the application and start executors.

# 7. Cluster Manager
Here is how the Cluster Manager decides where to put that new container:
Same Node (Most Common): If the worker node is completely healthy, but just one executor process crashed (for example, due to a bad configuration or a temporary memory spike), 
the Cluster Manager will simply launch a fresh executor container on that same worker node.

Different Node: If the original worker node is running out of memory entirely or entire Worker Node dies due to a hardware or network fault or failing its health checks, 
the Cluster Manager the Cluster Manager detects the offline machine, will spin up the new executor container  elsewhere on the cluster, and informs the Driver. 

The Cluster Manager decides: "I will allocate a container for you on Node A, B, or a brand-new Node C depending on where I have available hardware space."
If an entire Worker Node dies due to a hardware or network fault, the Cluster Manager detects the offline machine, allocates new resources elsewhere on the cluster, and informs the Driver 

# Key things to check before running it
# Environment: Ensure you have Apache Spark installed locally and your PATH includes the Spark bin directory, otherwise your terminal won't recognize spark-submit.
# SparkSession: Inside your my_job.py file, make sure you have initialized a SparkSession (e.g., spark = SparkSession.builder.getOrCreate()), 
or the script will just run as a regular Python file without utilizing Spark.

# 7. What happens when you run spark-submit?

# Imagine:

# spark-submit my_job.py

# Conceptually:

# 1. Submit application
#         ↓
# 2. Driver starts
#         ↓
# 3. Driver requests resources
#         ↓
# 4. Cluster manager allocates workers
#         ↓
# 5. Executors start
#         ↓
# 6. Driver creates execution plan
#         ↓
# 7. Driver creates stages
#         ↓
# 8. Stages create tasks
#         ↓
# 9. Tasks run on executors
#         ↓
# 10. Executors process data
#         ↓
# 11. Results written to storage

# MODES:
Local Mode ⚙️Driver Location: Both the Driver and the Executors run on a single machine inside a single Java Virtual Machine (JVM).
spark-submit --master local[*] my_job.py
Client Mode 💻Driver Location: The Driver runs on the client machine (e.g., your laptop or an edge gateway node) that submitted the command and executors on cluster worker node.
spark-submit --master yarn --deploy-mode client my_job.py
Cluster Mode 🏢Driver Location: The Driver runs inside the cluster on a worker node selected by the Cluster Manager. and also executors on cluster worker node
spark-submit --master yarn --deploy-mode cluster my_job.py

# The Apache Spark Web UI is available by default at port 4040 (http:#<driver-node>:4040) on the driver node to monitor a running cluster's tasks, executors, and storage usage.
Spark Cluster Monitoring & Web UI
Jobs Tab: View status, duration, and progress of all Spark jobs.
Stages Tab: Check summary metrics and task execution details for each stage.
Storage Tab: Inspect RDDs and DataFrames cached or stored in memory and disk across executors.
Executors Tab: Monitor memory and disk usage, task counts, and garbage collection metrics for each executor.
SQL Tab: Analyze execution plans, metrics, and durations for Spark SQL queries.
